In [ ]:
# Cell 1: Detect Kaggle / Colab GPU environment & install lightweight dependencies
import json, os, pathlib, sys, time, zipfile, subprocess

IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
OUT_DIR = pathlib.Path("/kaggle/working/outputs" if IS_KAGGLE else "/content/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Environment: {'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local')} | Out: {OUT_DIR}")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "diffusers", "transformers", "accelerate", "safetensors", "Pillow", "imageio[ffmpeg]"])
import torch
print("CUDA available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Cell 2: Embedded Job Specification (generated by cloud-gpu-video MCP)
JOB_SPEC = json.loads(r'''{
  "job_id": "ep01_lofi_hoodie_stickman",
  "created_at": "2026-11-26 09:38:40",
  "engine": "sdxl_lofi",
  "aspect_ratio": "9:16",
  "seed": 42,
  "character_lock": "Minimalist 2D lofi anime illustration, cute white round-headed stickman with crisp 3px black outlines, minimalist dot eyes and thin eyebrows, wearing an oversized dark charcoal hoodie, slim dark stick legs, and sneakers. Atmospheric warm/neon lighting, zero text.",
  "negative_prompt": "text, watermark, logo, blurry, deformed, extra limbs",
  "voiceover_script": "Three AM. Another fifty thousand dollar account gone. You walk out into the rain wondering if you should just quit. You sit in a quiet coffee shop, open your journal, and finally see the truth. It was never your strategy. It was trading without a daily loss limit. The next morning, you cut your risk to half a percent, take one clean setup, and close the laptop. Boring discipline is what finally gets you paid. Save this if you refuse to quit.",
  "caption": "3 AM blown account -> quiet coffee shop -> 0.5% discipline -> first funded payout. Save this if you refuse to quit.",
  "hashtags": [
    "#propfirm",
    "#fundedtrader",
    "#tradingpsychology",
    "#daytrading"
  ],
  "scenes": [
    {
      "scene_id": "scene_01",
      "prompt": "Sitting slumped at a wooden desk at 3 AM in a dimly lit bedroom, soft crimson glow from a monitor showing falling red candlesticks, rain on dark window behind.",
      "duration_sec": 5.0
    },
    {
      "scene_id": "scene_02",
      "prompt": "Walking slowly with hands in hoodie pockets along a rainy neon city sidewalk at night next to a glowing subway entrance, wet asphalt reflections.",
      "duration_sec": 5.0
    },
    {
      "scene_id": "scene_03",
      "prompt": "Sitting alone at a warm wooden window counter inside a cozy coffee shop at night, warm amber pendant lights, steaming coffee mug and open trading journal, rainy neon city outside window.",
      "duration_sec": 5.0
    },
    {
      "scene_id": "scene_04",
      "prompt": "Warm golden sunrise light pouring onto a clean wooden desk, calm posture watching a single monitor with a steady upward emerald green candlestick trend.",
      "duration_sec": 5.0
    },
    {
      "scene_id": "scene_05",
      "prompt": "Standing by a sunlit apartment window smiling quietly while holding a smartphone casting a soft emerald green glow, peaceful morning cityscape outside.",
      "duration_sec": 5.0
    }
  ]
}''')
print("Job:", JOB_SPEC["job_id"], "| Engine:", JOB_SPEC["engine"], "| Scenes:", len(JOB_SPEC["scenes"]))


In [ ]:
# Cell 3: Load GPU Pipeline & Generate All Scenes
import torch
from diffusers import AutoPipelineForText2Image
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.enable_model_cpu_offload()
NUM_STEPS = 4
GUIDANCE = 0.0
W, H = (768, 1344) if JOB_SPEC.get("aspect_ratio", "9:16") == "9:16" else (1344, 768)

CHAR_LOCK = JOB_SPEC.get("character_lock", "")
NEG = JOB_SPEC.get("negative_prompt", "text, watermark, signature, blurry, deformed, extra limbs, 3d render")
BASE_SEED = int(JOB_SPEC.get("seed", 42))
manifest = {"job_id": JOB_SPEC["job_id"], "engine": JOB_SPEC["engine"], "files": []}

for idx, sc in enumerate(JOB_SPEC["scenes"]):
    sid = sc.get("scene_id") or f"scene_{idx + 1:02d}"
    full_prompt = f"{CHAR_LOCK} {sc['prompt']}".strip()
    gen = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(BASE_SEED + idx)
    t0 = time.time()
    img = pipe(prompt=full_prompt, negative_prompt=NEG, width=W, height=H, num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE, generator=gen).images[0]
    out_path = OUT_DIR / f"{sid}.png"
    img.save(out_path)
    print(f"[OK] {sid} -> {out_path.name} ({time.time() - t0:.1f}s)")
    manifest["files"].append(out_path.name)

(OUT_DIR / "output_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
zip_path = OUT_DIR.parent / f"ep01_lofi_hoodie_stickman_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.glob("*")):
        zf.write(p, p.name)
print("PACKED:", zip_path, "| Total files:", len(manifest["files"]))


In [ ]:
# Cell 4: Optional Google Drive Auto-Backup on Colab
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drv_dir = pathlib.Path("/content/drive/MyDrive/AI_MONEY_SUITE_OUTPUTS/ep01_lofi_hoodie_stickman")
        drv_dir.mkdir(parents=True, exist_ok=True)
        import shutil
        for p in sorted(OUT_DIR.glob("*")):
            shutil.copy2(p, drv_dir / p.name)
        print("SYNCED TO GOOGLE DRIVE:", drv_dir)
    except Exception as exc:
        print("Drive mount skipped (download /content/ep01_lofi_hoodie_stickman_outputs.zip manually):", exc)
